# SLAY — AI Layout Evaluation Model
**Random Forest + SHAP | Flask REST API | ngrok tunnel**

## Quick Start
1. `Runtime → Run all` (Ctrl+F9)
2. Copy the **ngrok URL** from the last cell output
3. Paste it into `SLAY-v5-complete.html` where prompted
4. Click **Run AI** in the planner — predictions come from this model

## Research
- **Model**: Random Forest Classifier (sklearn) — 12 spatial features
- **XAI**: SHAP TreeExplainer — feature contribution values
- **Dataset**: 1690 synthetic event layout records (thesis Appendix A)
- **Accuracy**: 89.4% | F1: 89.9% | AUC-ROC: 0.941

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ═══════════════════════════════════════════════════════
!pip install flask flask-cors shap scikit-learn numpy pandas pyngrok --quiet

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2 — Generate synthetic training dataset
# ═══════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import shap, joblib, json

np.random.seed(42)
N = 1690

def generate_dataset(n):
    rows = []
    for _ in range(n):
        guests     = np.random.randint(20, 500)
        venue_area = np.random.randint(40, 1500)
        budget     = np.random.randint(50000, 5000000)
        exits      = np.random.randint(1, 8)
        ceil_h     = np.random.uniform(2.5, 10)
        has_stage  = np.random.choice([0,1], p=[.4,.6])
        catering   = np.random.choice(['buffet','plated','cocktail'])
        decor_lvl  = np.random.randint(1, 5)
        av_score   = np.random.uniform(0, 1)
        theme_coh  = np.random.uniform(0, 1)

        spg   = venue_area / guests
        bpg   = budget / guests
        dr    = min(guests / (venue_area * 0.5), 1.0)
        ec    = min(exits / 6, 1.0)
        bpg_n = max(0, min((bpg - 1000) / 3000, 1.0))

        # Feasibility label (rule-based ground truth)
        good = (
            spg >= 1.2 and
            exits >= 2 and
            bpg >= 1500 and
            dr <= 0.8 and
            theme_coh >= 0.4
        )

        # Add noise (10%)
        label = int(good) if np.random.random() > 0.10 else int(not good)

        rows.append({
            'space_per_guest':  round(min(spg, 5.0), 3),
            'density_ratio':    round(dr, 3),
            'exit_count_norm':  round(ec, 3),
            'budget_per_guest': round(bpg_n, 3),
            'table_coverage':   round(np.random.uniform(0.2, 0.8), 3),
            'aisle_ok':         int(spg > 1.4),
            'av_match':         round(av_score, 3),
            'theme_coherence':  round(theme_coh, 3),
            'decor_budget':     round(np.random.uniform(0, 1), 3),
            'catering_fit':     round(np.random.uniform(0.5, 1.0), 3),
            'accessibility':    round(np.random.uniform(0, 1), 3),
            'stage_ratio':      round(0.8 if has_stage else 1.0, 1),
            'feasible':         label
        })
    return pd.DataFrame(rows)

df = generate_dataset(N)
print(f'Dataset: {len(df)} rows | Feasible: {df.feasible.sum()} ({df.feasible.mean():.1%})')
df.head(3)

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3 — Train Random Forest model
# ═══════════════════════════════════════════════════════
FEATURES = [
    'space_per_guest', 'density_ratio', 'exit_count_norm', 'budget_per_guest',
    'table_coverage',  'aisle_ok',      'av_match',        'theme_coherence',
    'decor_budget',    'catering_fit',  'accessibility',   'stage_ratio'
]

X = df[FEATURES]
y = df['feasible']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(
    n_estimators   = 200,
    max_depth      = 12,
    min_samples_leaf = 4,
    random_state   = 42,
    n_jobs         = -1
)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

acc  = accuracy_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_proba)

print(f'Accuracy : {acc:.3f}  ({acc*100:.1f}%)')
print(f'F1 Score : {f1:.3f}')
print(f'AUC-ROC  : {auc:.3f}')

# Save model
joblib.dump(model, 'slay_rf_model.pkl')
print('Model saved: slay_rf_model.pkl')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 4 — SHAP explainer
# ═══════════════════════════════════════════════════════
explainer = shap.TreeExplainer(model)

# Test SHAP on first 5 samples
sample = X_test.iloc[:5]
shap_values = explainer.shap_values(sample)

print('SHAP values shape:', np.array(shap_values).shape)
print('Feature importances (model):')
for feat, imp in sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1]):
    bar = '█' * int(imp * 80)
    print(f'  {feat:25s} {imp:.4f}  {bar}')

# Save explainer
joblib.dump(explainer, 'slay_shap_explainer.pkl')
print('\nSHAP explainer saved.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 5 — Flask API
# ═══════════════════════════════════════════════════════
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

app = Flask(__name__)
CORS(app, origins='*')  # Allow all origins (required for file:// and localhost)

_model     = joblib.load('slay_rf_model.pkl')
_explainer = joblib.load('slay_shap_explainer.pkl')

FEATURE_NAMES = [
    'space_per_guest', 'density_ratio', 'exit_count_norm', 'budget_per_guest',
    'table_coverage',  'aisle_ok',      'av_match',        'theme_coherence',
    'decor_budget',    'catering_fit',  'accessibility',   'stage_ratio'
]

FEATURE_LABELS = {
    'space_per_guest':  'Space per guest',
    'density_ratio':    'Crowd density',
    'exit_count_norm':  'Emergency exits',
    'budget_per_guest': 'Budget per guest',
    'table_coverage':   'Table coverage',
    'aisle_ok':         'Aisle clearance',
    'av_match':         'AV suitability',
    'theme_coherence':  'Theme coherence',
    'decor_budget':     'Decor spending',
    'catering_fit':     'Catering fit',
    'accessibility':    'Accessibility',
    'stage_ratio':      'Stage planning',
}

def extract_features(data):
    """Convert raw event data dict to model feature vector."""
    guests      = max(1, float(data.get('guests', 100)))
    venue_area  = max(10, float(data.get('venue_area', 200)))
    budget      = max(0,  float(data.get('budget', 500000)))
    exits       = max(1,  int(data.get('exit_count', 2)))
    has_stage   = bool(data.get('has_stage', False))
    catering    = data.get('catering_style', 'buffet')
    decor_level = int(data.get('decor_level', 2))
    av_equip    = data.get('av_equipment', [])
    theme       = data.get('theme', 'classic')

    spg     = venue_area / guests
    bpg     = budget / guests
    dr      = min(guests / (venue_area * 0.5), 1.0)
    ec_norm = min(exits / 6, 1.0)
    bpg_n   = max(0, min((bpg - 1000) / 3000, 1.0))

    # AV match score by event type
    ev_type = data.get('event_type', 'wedding')
    has_pa  = 'pa_system' in av_equip
    has_mic = 'wireless_mic' in av_equip
    has_prj = 'projector' in av_equip or 'led_screen' in av_equip
    has_dj  = 'dj_booth' in av_equip
    if ev_type in ('seminar', 'corporate'):
        av_match = (int(has_prj) + int(has_mic) + int(has_pa)) / 3
    elif ev_type == 'wedding':
        av_match = (int(has_pa) + int(has_mic) + int(has_dj or has_prj)) / 3
    elif ev_type == 'birthday':
        av_match = (int(has_pa) + int(has_dj)) / 2
    else:
        av_match = 0.5

    # Theme coherence
    decor_types = data.get('decor_types', [])
    clashes = {'royal':['balloons'], 'minimalist':['balloons','drapes'], 'modern':['drapes']}
    tc = 0.75 - sum(0.15 for d in clashes.get(theme,[]) if d in decor_types)
    tc = max(0, tc)

    # Catering fit
    min_area = {'buffet':guests*1.5+20, 'plated':guests*1.2, 'cocktail':guests*0.8}
    cf = min(1.0, venue_area / max(1, min_area.get(catering, guests*1.5)))

    # Decor budget ratio
    db_ratio = {1:.12, 2:.25, 3:.38, 4:.52}.get(decor_level, .25)
    db_score = max(0, min(1, 1.0 - abs(db_ratio - 0.28) * 2.5))

    # Accessibility
    acc_needs = data.get('accessibility', [])
    acc = 1.0 if not acc_needs else (1.0 if exits >= 2 and spg >= 1.2 else 0.4)

    return [
        round(min(spg, 5.0), 4),
        round(dr, 4),
        round(ec_norm, 4),
        round(bpg_n, 4),
        round(float(data.get('table_coverage', 0.45)), 4),
        int(spg > 1.4),
        round(av_match, 4),
        round(tc, 4),
        round(db_score, 4),
        round(cf, 4),
        round(acc, 4),
        round(0.8 if has_stage else 1.0, 1),
    ]

@app.route('/health', methods=['GET'])
def health():
    return jsonify({ 'status':'ok', 'model':'RandomForest', 'features':12, 'version':'1.0' })

@app.route('/predict', methods=['POST', 'OPTIONS'])
def predict():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    try:
        data      = request.get_json(force=True)
        features  = extract_features(data)
        fv        = np.array(features).reshape(1, -1)

        # Prediction
        proba     = _model.predict_proba(fv)[0]
        score     = int(round(proba[1] * 100))
        feasible  = bool(proba[1] >= 0.5)

        # SHAP values
        shap_vals = _explainer.shap_values(fv)
        # shap_vals[1] = positive class contributions
        sv = shap_vals[1][0] if isinstance(shap_vals, list) else shap_vals[0]

        shap_out = []
        base_val = float(_explainer.expected_value[1] if isinstance(_explainer.expected_value, (list,np.ndarray)) else _explainer.expected_value)
        for i, feat in enumerate(FEATURE_NAMES):
            val = float(sv[i])
            shap_out.append({
                'feature':    feat,
                'label':      FEATURE_LABELS.get(feat, feat),
                'value':      round(val * 100, 2),
                'feature_val':features[i],
                'direction':  'positive' if val >= 0 else 'negative',
                'magnitude':  round(abs(val) * 100, 2),
            })
        shap_out.sort(key=lambda x: -x['magnitude'])

        return jsonify({
            'layout_score':  score,
            'feasible':      feasible,
            'probability':   round(float(proba[1]), 4),
            'shap_values':   shap_out[:8],
            'base_value':    round(base_val, 4),
            'features_used': dict(zip(FEATURE_NAMES, features)),
            'model':         'RandomForestClassifier',
            'n_estimators':  200,
        })
    except Exception as e:
        import traceback
        return jsonify({ 'error': str(e), 'trace': traceback.format_exc() }), 500

@app.route('/batch_predict', methods=['POST'])
def batch_predict():
    """Predict for multiple layouts at once."""
    try:
        layouts = request.get_json(force=True).get('layouts', [])
        results = []
        for layout in layouts:
            fv = np.array(extract_features(layout)).reshape(1,-1)
            proba = _model.predict_proba(fv)[0]
            results.append({ 'score': int(round(proba[1]*100)), 'feasible': bool(proba[1]>=0.5) })
        return jsonify({ 'results': results, 'count': len(results) })
    except Exception as e:
        return jsonify({ 'error': str(e) }), 500

@app.route('/feature_importance', methods=['GET'])
def feature_importance():
    imp = dict(zip(FEATURE_NAMES, _model.feature_importances_.tolist()))
    return jsonify({ 'importance': imp, 'labels': FEATURE_LABELS })

print('Flask app defined. Starting in background thread...')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 6 — Start Flask + ngrok tunnel
# ═══════════════════════════════════════════════════════
from pyngrok import ngrok, conf
import os

# If you have an ngrok auth token (free at ngrok.com), paste it below:
# ngrok.set_auth_token('YOUR_NGROK_TOKEN_HERE')

# Start Flask in background
def run_flask():
    app.run(port=5000, use_reloader=False, debug=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()

import time
time.sleep(2)  # Wait for Flask to start

# Open ngrok tunnel
public_url = ngrok.connect(5000).public_url

print('=' * 60)
print('SLAY AI API is LIVE')
print('=' * 60)
print(f'  Base URL  : {public_url}')
print(f'  Health    : {public_url}/health')
print(f'  Predict   : {public_url}/predict  (POST)')
print(f'  Batch     : {public_url}/batch_predict  (POST)')
print(f'  Features  : {public_url}/feature_importance  (GET)')
print('=' * 60)
print()
print('COPY THIS URL INTO SLAY HTML:')
print(f'  const AI_API_URL = "{public_url}";')
print()
print('Keep this tab open — closing it stops the API.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 7 — Test the API
# ═══════════════════════════════════════════════════════
import requests

test_payload = {
    'guests': 150, 'venue_area': 300, 'budget': 800000,
    'exit_count': 3, 'event_type': 'wedding', 'has_stage': True,
    'catering_style': 'buffet', 'theme': 'traditional_sl',
    'decor_level': 3, 'av_equipment': ['pa_system','wireless_mic','led_screen'],
    'accessibility': ['wheelchair'], 'decor_types': ['florals','arch'],
    'table_coverage': 0.48
}

r = requests.post(f'{public_url}/predict', json=test_payload)
result = r.json()

print(f'Layout score : {result["layout_score"]}/100')
print(f'Feasible     : {result["feasible"]}')
print(f'Probability  : {result["probability"]:.3f}')
print()
print('Top SHAP factors:')
for s in result['shap_values'][:5]:
    direction = '+' if s['direction']=='positive' else '-'
    bar = '█' * int(s['magnitude'] / 2)
    print(f'  {direction} {s["label"]:25s} {s["value"]:+.2f}  {bar}')